# FunnyBirds Recall Gap Analysis — Baseline vs CBM

Parallel to `recall.ipynb` (CUB baseline/CBM) but using the **FunnyBirds** synthetic dataset.

Compares:
- **Baseline** `resnet50_funnybirds/` — plain ResNet-50 trained on FunnyBirds species classification
- **CBM** `resnet50_cbm_funnybirds/` — Concept Bottleneck Model (same backbone + 26-concept bottleneck)

Extract baseline features (once) on Adroit login node:
```bash
python -m scripts.extract_features_funnybirds \\
    --funnybirds_root data/FunnyBirds \\
    --checkpoint      checkpoints_funnybirds/resnet50_funnybirds_best.pth \\
    --features_dir    features/resnet50_funnybirds \\
    --plain_resnet
```


For each attribute:

1. Train a linear probe on top of frozen visual features.
2. Evaluate the probe on held-out test images.
3. Group test images by species.
4. For pairs of species:
   - Subsample images so that both species have the same number of
     attribute-positive and attribute-negative examples.
   - Compute recall on attribute-positive images for each species.
5. Measure the recall gap between species.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


## 0. Configuration

- `FB` — FunnyBirds root
- `BASE_FEATS` — plain ResNet-50 features (extracted with `--plain_resnet`)
- `CBM_FEATS` — CBM backbone features (from `run_fb_cbm_adroit.sh`)


In [ ]:
import sys
ROOT = __import__('pathlib').Path('/scratch/network/cr7998/cv_emergence_project')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FB         = ROOT / 'data' / 'FunnyBirds'
BASE_FEATS = ROOT / 'features' / 'resnet50_funnybirds'
CBM_FEATS  = ROOT / 'features' / 'resnet50_cbm_funnybirds'

assert FB.exists(), f'Missing FunnyBirds folder: {FB}'
assert (FB / 'dataset_train.json').exists(), f'Missing dataset_train.json: {FB}'
for name, d in [('baseline', BASE_FEATS), ('cbm', CBM_FEATS)]:
    if not d.exists(): print(f'  [warn] {name} features not extracted: {d}')
    else:              print(f'  [ok]   {name}: {d}')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[config] device: {device}')


In [ ]:
def load_species_maps(fb_root: Path):
    """
    Load species ID→name mappings from FunnyBirds metadata/classes.csv.
    Mirrors load_species_maps(cub_root) in mcbm_recall_full.ipynb.
    """
    classes_csv = fb_root / "metadata" / "classes.csv"
    if not classes_csv.exists():
        raise FileNotFoundError(
            f"metadata/classes.csv not found. Run prepare_funnybirds_metadata.py first."
        )
    df = pd.read_csv(classes_csv)
    id2name  = dict(zip(df["class_id"], df["class_name"]))
    id2short = {k: v.replace("funnybird_", "FB") for k, v in id2name.items()}
    return id2name, id2short


In [ ]:
def load_meta(fb_root: Path) -> pd.DataFrame:
    """
    Load test-split image metadata with species information.
    Returns DataFrame with columns: image_id, class_id, species_id, species_name, is_train.
    Mirrors load_meta(cub_root) in mcbm_recall_full.ipynb.
    """
    images_csv = fb_root / "metadata" / "images.csv"
    if not images_csv.exists():
        raise FileNotFoundError(
            f"metadata/images.csv not found. Run prepare_funnybirds_metadata.py first."
        )
    df = pd.read_csv(images_csv)
    id2name, _ = load_species_maps(fb_root)
    df["species_id"]   = df["class_id"]
    df["species_name"] = df["class_id"].map(id2name)
    return df


In [ ]:
def load_image_attr_labels_robust(fb_root: Path) -> pd.DataFrame:
    """
    Load per-image binary concept labels from metadata/image_concepts_binary.csv.
    Mirrors load_image_attr_labels_robust(cub_root) in mcbm_recall_full.ipynb.

    Returns a long-form DataFrame with columns:
        image_id, attr_id (concept_id), attr_name, is_present, certainty
    certainty is always 1 for FunnyBirds (ground-truth, no annotation noise).
    """
    concepts_csv = fb_root / "metadata" / "image_concepts_binary.csv"
    if not concepts_csv.exists():
        raise FileNotFoundError(
            f"metadata/image_concepts_binary.csv not found. "
            f"Run prepare_funnybirds_metadata.py first."
        )
    wide = pd.read_csv(concepts_csv)
    concept_cols = [c for c in wide.columns if c != "image_id"]
    long = wide.melt(id_vars="image_id", value_vars=concept_cols,
                     var_name="attr_name", value_name="is_present")
    long["attr_id"] = long.groupby("attr_name", sort=False).ngroup()
    long["is_present"] = long["is_present"].astype(int)
    long["certainty"] = 1  # FunnyBirds: ground-truth labels, always certain
    return long[["image_id", "attr_id", "attr_name", "is_present", "certainty"]]


In [ ]:
def load_attr_maps(fb_root: Path):
    """
    Load concept names from metadata/concepts.csv.
    Mirrors load_attr_maps(attr_txt) in mcbm_recall_full.ipynb.

    Returns:
        attr_id_to_name : dict {concept_id -> concept_name}
        attr_name_to_id : dict {concept_name -> concept_id}
    """
    concepts_csv = fb_root / "metadata" / "concepts.csv"
    if not concepts_csv.exists():
        raise FileNotFoundError(
            f"metadata/concepts.csv not found. Run prepare_funnybirds_metadata.py first."
        )
    df = pd.read_csv(concepts_csv)
    id2name  = dict(zip(df["concept_id"], df["concept_name"]))
    name2id  = dict(zip(df["concept_name"], df["concept_id"]))
    return id2name, name2id


## 1. Concept selection

FunnyBirds has exactly **26 binary concepts** (one-hot over part variants):

| Part  | Variants | Concept dims |
|-------|----------|-------------|
| beak  | 4        | beak_0 .. beak_3 |
| eye   | 3        | eye_0  .. eye_2  |
| wing  | 6        | wing_0 .. wing_5 |
| foot  | 4        | foot_0 .. foot_3 |
| tail  | 9        | tail_0 .. tail_8 |

Each species uses exactly one variant per part → concept vector is one-hot per part group.
No 10-90% prevalence filter needed: by construction each variant is used by 50/n_variants species.


In [ ]:
# FunnyBirds: all 26 concepts (no prevalence filter needed)
from datasets.funnybirds_dataset import concept_names as _fb_concept_names
ATTR_LIST = _fb_concept_names()
print(f"FunnyBirds concepts ({len(ATTR_LIST)}): {ATTR_LIST}")


In [ ]:
# ── FunnyBirds ground truth: class-concept matrix ────────────────────────────
# Every row is a species; every column is one of 26 part-variant concepts.
# This matrix is EXACT (no annotation noise) — the key FunnyBirds advantage.

from datasets.funnybirds_dataset import FunnyBirdsDataset
_fb_ds = FunnyBirdsDataset(FB, split="train")
class_concept_matrix, _ = _fb_ds.get_class_concept_matrix()

# Display as DataFrame
import pandas as pd
from datasets.funnybirds_dataset import concept_names as _cnames, PART_VARIANTS
cc_df = pd.DataFrame(
    class_concept_matrix.numpy(),
    columns=_cnames(),
    index=[f"funnybird_{i:02d}" for i in range(class_concept_matrix.shape[0])],
)
print(f"Class-concept matrix shape: {cc_df.shape}")
print("Each row sums to 5 (one variant per part per species):")
print(cc_df.sum(axis=1).value_counts())
print("\nFirst 5 species:")
cc_df.head()


In [ ]:
# Load metadata, concept labels, and attribute maps
meta           = load_meta(FB)
img_attr_long  = load_image_attr_labels_robust(FB)
attr_id_to_name, attr_name_to_id = load_attr_maps(FB)

# attr_df: one row per concept — mirrors CUB attr_df used downstream
attr_df = pd.DataFrame({
    "attr_name": list(attr_name_to_id.keys()),
    "attr_id":   list(attr_name_to_id.values()),
})

# spname: species id → display name (used by matched_pair_eval / fb_pair_eval)
id2name, _ = load_species_maps(FB)
spname = lambda sid: id2name.get(int(sid), f"funnybird_{int(sid):02d}")

print(f"meta:          {len(meta)} images  ({meta['is_train'].sum()} train, {(meta['is_train']==0).sum()} test)")
print(f"img_attr_long: {len(img_attr_long)} rows, {img_attr_long['attr_name'].nunique()} concepts")
print(f"attr_df:       {len(attr_df)} concepts")
print(f"Concepts:      {list(attr_name_to_id.keys())}")


In [ ]:
# What this cell does:
# - For a given attribute_id, merges:
#   meta (image->species, split) with attribute labels (image->y)
# - Produces a clean table with y in {0,1}. where y is whether the attribut below is present or not.
# Why it matters:
# - This is the ground-truth label table used for training and evaluation.

def build_attr_labeled_df(meta: pd.DataFrame,
                          img_attr_long: pd.DataFrame,
                          attr_id: int,
                          min_certainty: int = 1) -> pd.DataFrame:
    """
    Returns dataframe with:
      image_id, species_id, species_name, is_train, y, certainty
    Only keeps annotations with certainty >= min_certainty.
    """
    sub = img_attr_long[img_attr_long["attr_id"] == int(attr_id)].copy()
    sub = sub[sub["certainty"] >= int(min_certainty)].copy()

    out = meta.merge(sub[["image_id", "is_present", "certainty"]], on="image_id", how="inner")
    out = out.rename(columns={"is_present": "y"})
    out["y"] = out["y"].astype(int)
    return out[["image_id", "species_id", "species_name", "is_train", "y", "certainty"]]

print("Defined:", "build_attr_labeled_df")

# sanity check on one attribute
attr_name = "beak_0"
aid = attr_name_to_id[attr_name]
lab = build_attr_labeled_df(meta, img_attr_long, aid, min_certainty=1)
print("Attribute:", attr_name, "rows labeled:", len(lab), "pos rate:", lab.y.mean())
lab.head()


In [ ]:
# - Loads a feature tensor from disk and converts it to float32 torch.Tensor.

def safe_torch_load(path: Path):
    """
    Uses weights_only=True if supported to reduce pickle risk warnings.
    """
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")

def load_features(feat_dir: Path, layer: str, split: str) -> torch.Tensor:
    """
    Loads feature tensor saved as {layer}_{split}.pt from feat_dir.
    """
    p = feat_dir / f"{layer}_{split}.pt"
    assert p.exists(), f"Missing: {p}"
    X = safe_torch_load(p)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)
    return X.float()

In [ ]:
import numpy as np
import torch

def to_1d_int_array(x):
    """Convert tensor/list/np array to 1D int numpy array."""
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.array(x)
    x = x.reshape(-1)
    return x.astype(int)

def load_split_order(feat_dir, split):
    p = feat_dir / f"labels_{split}.pt"
    assert p.exists(), f"Missing: {p}"
    t = torch.load(p, map_location="cpu", weights_only=True)

    assert isinstance(t, dict), f"Expected dict in {p}, got {type(t)}"
    assert "image_ids" in t, f"{p} missing 'image_ids' key; has {list(t.keys())}"

    ids = to_1d_int_array(t["image_ids"])
    kind = infer_kind(ids)
    return kind, ids


def infer_kind(arr):
    # Heuristic:
    # - species ids: 1..200 (sometimes 0..199)
    # - image ids: 1..11788
    if arr.max() <= 200 and arr.min() >= 0:
        return "species_id_like"
    if arr.max() > 200:
        return "image_id_like"
    return "unknown"


## 2. Loading pre-computed features

- **Baseline** (`resnet50_funnybirds/`) — plain ResNet-50, no concept supervision
- **CBM** (`resnet50_cbm_funnybirds/`) — CBM backbone (sigmoid concept bottleneck)

Same 18 probe points: conv1 → layer1.0..layer4.2 → avgpool.


In [ ]:
LAYER = 'layer4.0'

# Use whichever feature dir is available as primary reference
_ref = CBM_FEATS if CBM_FEATS.exists() else BASE_FEATS
assert _ref.exists(), 'Need at least one feature set. Extract features first.'

base_kind_tr, base_ids_tr = load_split_order(BASE_FEATS, 'train') if BASE_FEATS.exists() else (None, None)
base_kind_te, base_ids_te = load_split_order(BASE_FEATS, 'test')  if BASE_FEATS.exists() else (None, None)
cbm_kind_tr,  cbm_ids_tr  = load_split_order(CBM_FEATS,  'train') if CBM_FEATS.exists()  else (None, None)
cbm_kind_te,  cbm_ids_te  = load_split_order(CBM_FEATS,  'test')  if CBM_FEATS.exists()  else (None, None)

print('BASE_FEATS available:', BASE_FEATS.exists())
print('CBM_FEATS  available:', CBM_FEATS.exists())


In [ ]:
# What this cell does:
# - Aligns features (in feature row order) to attribute labels by image_id.
# - Produces X_aligned and df_aligned with the same ordering.


def align_features_and_labels(X_split: torch.Tensor,
                              image_ids_in_feature_order: np.ndarray,
                              labeled_df_split: pd.DataFrame):
    """
    Inputs:
      X_split: feature tensor of shape [N, D]
      image_ids_in_feature_order: length N, image_id for each row of X_split
      labeled_df_split: dataframe with at least columns [image_id, y, species_id, species_name]

    Output:
      X_aligned: features for images that have labels
      df_aligned: same rows, same order, includes y and species info
    """
    labeled = labeled_df_split.set_index("image_id")[["y", "species_id", "species_name"]]

    keep_idx = []
    rows = []
    for i, img_id in enumerate(image_ids_in_feature_order):
        img_id = int(img_id)
        if img_id in labeled.index:
            keep_idx.append(i)
            y, sid, sname = labeled.loc[img_id]
            rows.append((img_id, int(sid), str(sname), int(y)))

    X_aligned = X_split[keep_idx]
    df_aligned = pd.DataFrame(rows, columns=["image_id", "species_id", "species_name", "y"])
    return X_aligned, df_aligned

## 3. Linear probe — training and evaluation

For each attribute we train a **1-layer linear probe** on frozen backbone features
at layer `LAYER` (default: `layer4.0`).

| Setting | Value |
|---|---|
| Loss | `BCEWithLogitsLoss` with `pos_weight` (class-imbalance correction) |
| Optimiser | `AdamW`, lr = 1e-2, wd = 1e-4 |
| Epochs | 25 |
| Batch size | 512 |

The probe is the *measurement instrument*: if an attribute is linearly decodable
from the backbone features, the probe will detect it.
The matched-pair test (§5) then asks whether detection recall differs across species
**even when attribute prevalence is held constant** — i.e. whether the probe
exploits species identity as a shortcut (entanglement).

In [ ]:
# What this cell does:
# - Defines a linear probe (single linear layer).
# - Trains it using BCEWithLogitsLoss with mild class-imbalance handling.
# Why it matters:
# - Probe is the measurement instrument for "is the attribute encoded in features?"

class LinearProbe(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.lin = nn.Linear(d, 1)

    def forward(self, x):
        return self.lin(x).squeeze(-1)

def train_probe(Xtr: torch.Tensor, ytr: np.ndarray,
                seed=0, lr=1e-2, wd=1e-4, epochs=25, batch=512):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    Xtr = Xtr.to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32, device=device)

    probe = LinearProbe(Xtr.shape[1]).to(device)
    opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=wd)

    pos = float(ytr_t.mean().item())
    pos_weight = torch.tensor([(1 - pos) / pos], device=device) if 0 < pos < 1 else torch.tensor([1.0], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    n = Xtr.shape[0]
    for _ in range(epochs):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            logits = probe(Xtr[idx])
            loss = loss_fn(logits, ytr_t[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()

    return probe

@torch.no_grad()
def predict_probs(probe: nn.Module, X: torch.Tensor, batch=4096) -> np.ndarray:
    probe.eval()
    probs = []
    for i in range(0, X.shape[0], batch):
        xb = X[i:i+batch].to(device)
        logits = probe(xb)
        probs.append(torch.sigmoid(logits).detach().cpu())
    return torch.cat(probs, dim=0).numpy()

print("Defined:", "LinearProbe", "train_probe", "predict_probs")


1. Identify species that have both positive and negative examples.
2. For each pair:
   - Subsample so both species have identical numbers of positives and negatives.
3. Compute recall on positive examples for each species.
4. Measure the absolute recall difference.

In [ ]:
# What this cell does:
# - Finds species that have enough positives and negatives for the chosen attribute.
# - Samples many species pairs.
# - For each pair, subsamples to match prevalence exactly and computes recall on positives.
# Why it matters:
# - This isolates species-specific differences even when prevalence is controlled perfectly.
#
# FunnyBirds note: in FunnyBirds every attribute is fixed per species (all-positive or
# all-negative — no within-species variation).  make_candidate_pairs therefore always
# returns 0 pairs.  make_candidate_pairs_fb instead pairs two ALL-POSITIVE species and
# measures whether the probe recalls the attribute equally well for both, giving the
# same entanglement signal without requiring within-species negatives.

def make_candidate_pairs(df_test: pd.DataFrame, min_each=10, max_pairs=200, seed=0):
    """
    Returns list of tuples (sid_A, sid_B, mpos, mneg) where:
      mpos = min(posA, posB)
      mneg = min(negA, negB)
    and both are >= min_each.
    """
    g = df_test.groupby("species_id")["y"].agg(["count", "sum"]).rename(columns={"sum": "pos"})
    g["neg"] = g["count"] - g["pos"]
    ok = g[(g["pos"] >= min_each) & (g["neg"] >= min_each)]
    sids = ok.index.to_list()

    rng = np.random.default_rng(seed)
    pairs = []
    if len(sids) < 2:
        return pairs

    for _ in range(max_pairs * 10):
        a, b = rng.choice(sids, size=2, replace=False)
        mpos = int(min(ok.loc[a, "pos"], ok.loc[b, "pos"]))
        mneg = int(min(ok.loc[a, "neg"], ok.loc[b, "neg"]))
        if mpos >= min_each and mneg >= min_each:
            pairs.append((int(a), int(b), mpos, mneg))
        if len(pairs) >= max_pairs:
            break
    return pairs


def make_candidate_pairs_fb(df_test: pd.DataFrame, min_pos: int = 3, max_pairs: int = 200, seed: int = 0):
    """
    FunnyBirds variant: pairs all-positive species (prevalence >= 0.9, n_pos >= min_pos).

    In FunnyBirds every species is either entirely positive or entirely negative for a
    given attribute, so the standard make_candidate_pairs (which requires within-species
    positives AND negatives) never finds valid pairs.  This function instead enumerates
    pairs of all-positive species and returns (sid_A, sid_B, mpos) 3-tuples where
    mpos = min(n_pos_A, n_pos_B).

    The entanglement signal: if the probe fires differently on images from species A vs B
    even though both truly have the attribute, that indicates species-identity leakage.
    """
    from itertools import combinations as _combinations
    g = df_test.groupby("species_id")["y"].agg(["count", "sum"]).rename(columns={"sum": "pos"})
    g["prev"] = g["pos"] / g["count"]
    ok = g[(g["pos"] >= min_pos) & (g["prev"] >= 0.9)]
    sids = ok.index.tolist()

    pairs = []
    for a, b in _combinations(sids, 2):
        mpos = int(min(ok.loc[a, "pos"], ok.loc[b, "pos"]))
        pairs.append((int(a), int(b), mpos))
        if len(pairs) >= max_pairs:
            break
    return pairs


def matched_pair_eval(df_test: pd.DataFrame, probs: np.ndarray, sid_A: int, sid_B: int,
                      mpos: int, mneg: int, seed=0, thr=0.5):
    """
    Subsamples to:
      mpos positives + mneg negatives from each species.
    Computes recall on positive examples only for each species subset.
    """
    df = df_test.copy()
    df["prob"] = probs

    A = df[df.species_id == sid_A]
    B = df[df.species_id == sid_B]

    A_pos, A_neg = A[A.y == 1], A[A.y == 0]
    B_pos, B_neg = B[B.y == 1], B[B.y == 0]

    A_s = pd.concat([A_pos.sample(mpos, random_state=seed), A_neg.sample(mneg, random_state=seed)])
    B_s = pd.concat([B_pos.sample(mpos, random_state=seed), B_neg.sample(mneg, random_state=seed)])

    def recall_pos(d):
        pos = d[d.y == 1]
        pred = (pos.prob.values >= thr).astype(int)
        return float((pred == 1).mean()) if len(pos) else np.nan

    recA = recall_pos(A_s)
    recB = recall_pos(B_s)

    return {
        "sid_A": sid_A,
        "sid_B": sid_B,
        "species_A": spname(sid_A),
        "species_B": spname(sid_B),
        "npos": int(mpos),
        "nneg": int(mneg),
        "recall_A": float(recA),
        "recall_B": float(recB),
        "gap": float(abs(recA - recB)),
    }


def fb_pair_eval(df_test: pd.DataFrame, probs: np.ndarray, sid_A: int, sid_B: int,
                 mpos: int, seed: int = 0, thr: float = 0.5):
    """
    FunnyBirds variant of matched_pair_eval for all-positive species pairs.

    Samples mpos images (with replacement for bootstrap) from each all-positive species
    and computes recall = fraction classified positive.  The gap is the absolute
    difference in recall between species A and B.

    No negatives are drawn — both species are all-positive for this attribute.
    """
    df = df_test.copy()
    df["prob"] = probs

    A = df[(df.species_id == sid_A) & (df.y == 1)]
    B = df[(df.species_id == sid_B) & (df.y == 1)]

    # Sample with replacement (bootstrap); cap at actual size on the first call
    n_A = min(mpos, len(A))
    n_B = min(mpos, len(B))
    A_s = A.sample(n_A, random_state=seed, replace=(n_A < mpos))
    B_s = B.sample(n_B, random_state=seed, replace=(n_B < mpos))

    def recall_pos(d):
        pred = (d.prob.values >= thr).astype(int)
        return float(pred.mean()) if len(d) > 0 else np.nan

    recA = recall_pos(A_s)
    recB = recall_pos(B_s)

    return {
        "sid_A": sid_A,
        "sid_B": sid_B,
        "species_A": spname(sid_A),
        "species_B": spname(sid_B),
        "npos": int(mpos),
        "nneg": 0,
        "recall_A": float(recA) if recA is not None else np.nan,
        "recall_B": float(recB) if recB is not None else np.nan,
        "gap": float(abs(recA - recB)) if (recA is not None and recB is not None) else np.nan,
    }


def matched_pair_bootstrap_summary(
    df_te: pd.DataFrame,
    probs: np.ndarray,
    pairs,
    *,
    thr=0.5,
    B=300,
):
    """
    For each candidate (sid_A, sid_B, mpos, mneg):
      - run matched_pair_eval B times (each time resampling matched subsets)
      - summarize distribution of 'gap' across bootstrap runs

    Returns:
      res_long: one row per (pair, bootstrap_run) with recall_A, recall_B, gap
      pair_summary: one row per pair with gap_mean/std/CI/p and stability metrics
    """
    matched_cols = [
        "sid_A", "sid_B", "species_A", "species_B",
        "npos", "nneg", "recall_A", "recall_B", "gap", "boot_id"
    ]
    summary_cols = [
        "sid_A","sid_B","species_A","species_B",
        "npos","nneg",
        "gap_mean","gap_std","gap_ci_lo","gap_ci_hi","gap_p",
        "gap_ci_width","gap_snr","gap_norm","n_runs"
    ]

    if pairs is None or len(pairs) == 0:
        return pd.DataFrame(columns=matched_cols), pd.DataFrame(columns=summary_cols)

    rows = []
    for (a, b, mpos, mneg) in pairs:
        for boot_id in range(B):
            r = matched_pair_eval(
                df_te, probs,
                sid_A=int(a), sid_B=int(b),
                mpos=int(mpos), mneg=int(mneg),
                seed=int(boot_id), thr=float(thr)
            )
            r["boot_id"] = int(boot_id)
            rows.append(r)

    res_long = pd.DataFrame(rows)

    if res_long.empty:
        return res_long, pd.DataFrame(columns=summary_cols)

    def _ci_lo(x): return bootstrap_ci(x)[0]
    def _ci_hi(x): return bootstrap_ci(x)[1]

    pair_summary = (
        res_long.groupby(["sid_A","sid_B","species_A","species_B"], as_index=False)
                .agg(
                    npos=("npos","min"),
                    nneg=("nneg","min"),
                    gap_mean=("gap","mean"),
                    gap_std=("gap","std"),
                    gap_ci_lo=("gap", _ci_lo),
                    gap_ci_hi=("gap", _ci_hi),
                    gap_p=("gap", bootstrap_p_value),
                    n_runs=("gap","size"),
                )
    )

    EPS = 1e-12
    pair_summary["gap_ci_width"] = pair_summary["gap_ci_hi"] - pair_summary["gap_ci_lo"]

    # Stability metric: big gap relative to variability
    pair_summary["gap_snr"] = pair_summary["gap_mean"] / (pair_summary["gap_std"].fillna(0.0) + EPS)

    # Normalized effect size on 0..1 gap scale (gap itself already in [0,1])
    pair_summary["gap_norm"] = pair_summary["gap_mean"]

    pair_summary = pair_summary.sort_values("gap_mean", ascending=False).reset_index(drop=True)

    return res_long, pair_summary


def fb_bootstrap_summary(
    df_te: pd.DataFrame,
    probs: np.ndarray,
    pairs,
    *,
    thr: float = 0.5,
    B: int = 300,
):
    """
    FunnyBirds variant of matched_pair_bootstrap_summary.

    Accepts 3-tuple pairs (sid_A, sid_B, mpos) from make_candidate_pairs_fb and uses
    fb_pair_eval (all-positive species, bootstrap with replacement).  Returns the same
    column schema as matched_pair_bootstrap_summary so downstream code is unchanged.
    """
    matched_cols = [
        "sid_A", "sid_B", "species_A", "species_B",
        "npos", "nneg", "recall_A", "recall_B", "gap", "boot_id"
    ]
    summary_cols = [
        "sid_A","sid_B","species_A","species_B",
        "npos","nneg",
        "gap_mean","gap_std","gap_ci_lo","gap_ci_hi","gap_p",
        "gap_ci_width","gap_snr","gap_norm","n_runs"
    ]

    if pairs is None or len(pairs) == 0:
        return pd.DataFrame(columns=matched_cols), pd.DataFrame(columns=summary_cols)

    rows = []
    for (a, b, mpos) in pairs:
        for boot_id in range(B):
            r = fb_pair_eval(
                df_te, probs,
                sid_A=int(a), sid_B=int(b),
                mpos=int(mpos),
                seed=int(boot_id), thr=float(thr),
            )
            r["boot_id"] = int(boot_id)
            rows.append(r)

    res_long = pd.DataFrame(rows)

    if res_long.empty:
        return res_long, pd.DataFrame(columns=summary_cols)

    res_long = res_long.dropna(subset=["gap"])
    if res_long.empty:
        return res_long, pd.DataFrame(columns=summary_cols)

    def _ci_lo(x): return bootstrap_ci(x)[0]
    def _ci_hi(x): return bootstrap_ci(x)[1]

    pair_summary = (
        res_long.groupby(["sid_A","sid_B","species_A","species_B"], as_index=False)
                .agg(
                    npos=("npos","min"),
                    nneg=("nneg","min"),
                    gap_mean=("gap","mean"),
                    gap_std=("gap","std"),
                    gap_ci_lo=("gap", _ci_lo),
                    gap_ci_hi=("gap", _ci_hi),
                    gap_p=("gap", bootstrap_p_value),
                    n_runs=("gap","size"),
                )
    )

    EPS = 1e-12
    pair_summary["gap_ci_width"] = pair_summary["gap_ci_hi"] - pair_summary["gap_ci_lo"]
    pair_summary["gap_snr"] = pair_summary["gap_mean"] / (pair_summary["gap_std"].fillna(0.0) + EPS)
    pair_summary["gap_norm"] = pair_summary["gap_mean"]
    pair_summary = pair_summary.sort_values("gap_mean", ascending=False).reset_index(drop=True)

    return res_long, pair_summary


def species_recall_prevalence_table(df_te: pd.DataFrame, probs: np.ndarray, thr=0.5) -> pd.DataFrame:
    """
    Per-species table on the TEST set:
      - n, n_pos, n_neg
      - prevalence = n_pos / n
      - tp = # of positives predicted positive
      - recall = tp / n_pos
      - precision = tp / n_pred_pos   (optional but cheap and often useful)
    """
    df = df_te[["species_id", "species_name", "y"]].copy()
    df["prob"] = np.asarray(probs, dtype=float)
    df["pred"] = (df["prob"] >= thr).astype(int)

    # Basic counts per species
    g = (df.groupby(["species_id", "species_name"], as_index=False)
           .agg(
               n=("y", "size"),
               n_pos=("y", "sum"),
               n_pred_pos=("pred", "sum"),
           ))
    g["n_neg"] = g["n"] - g["n_pos"]
    g["prevalence"] = g["n_pos"] / g["n"]

    # True positives per species (only among y==1)
    tp = (df[df["y"] == 1]
            .groupby(["species_id", "species_name"])["pred"]
            .sum()
            .reset_index(name="tp"))

    out = g.merge(tp, on=["species_id", "species_name"], how="left")
    out["tp"] = out["tp"].fillna(0).astype(int)

    # Recall: tp / n_pos (handle n_pos==0)
    out["recall"] = np.where(out["n_pos"] > 0, out["tp"] / out["n_pos"], np.nan)

    # Precision: tp / n_pred_pos (handle n_pred_pos==0)
    out["precision"] = np.where(out["n_pred_pos"] > 0, out["tp"] / out["n_pred_pos"], np.nan)

    out = out.sort_values(["n"], ascending=False).reset_index(drop=True)
    return out


def add_species_bootstrap_ci(df_te: pd.DataFrame, probs: np.ndarray, thr=0.5, B=300, min_pos_for_ci=1) -> pd.DataFrame:
    """
    Adds bootstrap CIs for per-species recall.
    Bootstraps *within each species* by resampling that species' test images with replacement.

    Outputs new columns:
      - recall_ci_lo, recall_ci_hi
      - recall_bs_mean (bootstrap mean; usually close to recall)
      - recall_ci_width
    """
    df = df_te[["species_id", "species_name", "y"]].copy()
    df["prob"] = np.asarray(probs, dtype=float)

    rows = []
    rng = np.random.default_rng(0)

    for (sid, sname), d in df.groupby(["species_id", "species_name"]):
        d = d.reset_index(drop=True)
        n = len(d)
        n_pos = int(d["y"].sum())

        # If no positives, recall undefined (and CI meaningless)
        if n_pos < min_pos_for_ci:
            rows.append({
                "species_id": int(sid),
                "species_name": str(sname),
                "recall_bs_mean": np.nan,
                "recall_ci_lo": np.nan,
                "recall_ci_hi": np.nan,
                "recall_ci_width": np.nan,
                "B": int(B),
            })
            continue

        vals = []
        for b in range(B):
            idx = rng.integers(0, n, size=n)  # resample rows with replacement
            s = d.iloc[idx]

            pos = s[s["y"] == 1]
            if len(pos) == 0:
                vals.append(np.nan)
                continue
            pred_pos = (pos["prob"].to_numpy() >= thr).astype(int)
            vals.append(float(pred_pos.mean()))

        vals = np.asarray(vals, dtype=float)
        lo, hi = bootstrap_ci(vals, alpha=0.05)
        rows.append({
            "species_id": int(sid),
            "species_name": str(sname),
            "recall_bs_mean": float(np.nanmean(vals)) if len(vals) > 0 else np.nan,
            "recall_ci_lo": lo,
            "recall_ci_hi": hi,
            "recall_ci_width": (hi - lo) if (np.isfinite(lo) and np.isfinite(hi)) else np.nan,
            "B": int(B),
        })

    ci_df = pd.DataFrame(rows)
    base = species_recall_prevalence_table(df_te, probs, thr=thr)
    out = base.merge(ci_df, on=["species_id", "species_name"], how="left")
    return out


In [ ]:
def bootstrap_ci(x, alpha=0.05):
    """
    Percentile bootstrap CI for a 1D array x.
    Returns (lo, hi). If x is empty or all-nan, returns (nan, nan).
    """
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return (np.nan, np.nan)
    lo = np.quantile(x, alpha/2)
    hi = np.quantile(x, 1 - alpha/2)
    return (float(lo), float(hi))

def bootstrap_p_value(values, null=0.0):
    """
    Two-sided bootstrap p-value for H0: E[value] == null
    Using bootstrap distribution of the statistic itself.

    p = 2 * min(P(value <= null), P(value >= null))
    """
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if v.size == 0:
        return np.nan
    p_lo = np.mean(v <= null)
    p_hi = np.mean(v >= null)
    return float(2.0 * min(p_lo, p_hi))

def safe_div(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = np.full_like(a, np.nan, dtype=float)
    m = b != 0
    out[m] = a[m] / b[m]
    return out


In [ ]:
def run_one_attribute(
    attr_name: str,
    feat_dir: Path,
    split_order_kind_train: str,
    split_order_train: np.ndarray,
    split_order_kind_test: str,
    split_order_test: np.ndarray,
    layer: str,
    *,
    min_certainty: int = 1,
    thr: float = 0.5,
    epochs: int = 25,
    min_each: int = 10,
    n_pairs: int = 200,
    B_gap: int = 100,
    B_species: int = 100,
    use_balanced_acc: bool = False,
):
    """
    Runs the full pipeline for ONE attribute and ONE model's features:
      1) Build labeled train/test sets for this attribute (certainty filtering)
      2) Load train/test features for the chosen layer
      3) Align features to labels by image_id using split order arrays
      4) Train a linear probe on train features
      5) Predict probabilities on test features
      6) Build per-species prevalence+recall table (and bootstrap CI for recall)
      7) (Optional) Generate candidate species pairs for matched evaluation
      8) (Optional) Bootstrap matched-pair recall gap per pair (CI + p-value + stability metrics)
    """

    # Alignment requires feature order indexed by image_id.
    assert split_order_kind_train == "image_id_like" and split_order_kind_test == "image_id_like", (
        "Cannot align features to attribute labels because labels_{split}.pt is not image_id-like.\n"
        "If you hit this, we need to read the dataset ordering from your extractor code."
    )

    # Build per-image labels for this attribute.
    aid = attr_name_to_id[attr_name]
    lab = build_attr_labeled_df(meta, img_attr_long, aid, min_certainty=min_certainty)
    lab_train = lab[lab["is_train"] == 1].copy()
    lab_test  = lab[lab["is_train"] == 0].copy()

    # Load precomputed features.
    Xtr_all = load_features(feat_dir, layer, "train")
    Xte_all = load_features(feat_dir, layer, "test")

    # Align labeled images to feature tensor order.
    Xtr, df_tr = align_features_and_labels(Xtr_all, split_order_train, lab_train)
    Xte, df_te = align_features_and_labels(Xte_all, split_order_test,  lab_test)

    ytr = df_tr["y"].astype(int).to_numpy()
    yte = df_te["y"].astype(int).to_numpy()

    # Train probe + predict probabilities on test.
    probe = train_probe(Xtr, ytr, seed=0, epochs=epochs)
    probs = predict_probs(probe, Xte)

    # Species table (point estimates + bootstrap CI).
    if B_species > 0:
        species_table = add_species_bootstrap_ci(df_te, probs, thr=thr, B=B_species)
    else:
        species_table = species_recall_prevalence_table(df_te, probs, thr=thr)

    # Overall accuracy — plain or balanced (controlled by use_balanced_acc).
    # Balanced: 0.5*(TPR + TNR), not inflated by class imbalance.
    if len(yte) == 0:
        test_acc = np.nan
    elif use_balanced_acc:
        pred_bin = (probs >= thr).astype(int)
        tp = int(((pred_bin == 1) & (yte == 1)).sum())
        tn = int(((pred_bin == 0) & (yte == 0)).sum())
        fp = int(((pred_bin == 1) & (yte == 0)).sum())
        fn = int(((pred_bin == 0) & (yte == 1)).sum())
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        test_acc = 0.5 * (tpr + tnr)
    else:
        test_acc = float(((probs >= thr).astype(int) == yte).mean())

    # Matched pairs (optional).
    if n_pairs is None or int(n_pairs) <= 0:
        res_long = pd.DataFrame()
        pair_summary = pd.DataFrame()
        pairs = []
    else:
        pairs = make_candidate_pairs(df_te, min_each=min_each, max_pairs=n_pairs, seed=0)
        if len(pairs) > 0:
            res_long, pair_summary = matched_pair_bootstrap_summary(
                df_te, probs, pairs, thr=thr, B=B_gap
            )
        else:
            # FunnyBirds fallback: attributes are fixed per species (all-positive or
            # all-negative), so the standard paired test never finds valid pairs.
            # Instead, pair all-positive species and measure cross-species recall gap.
            fb_pairs = make_candidate_pairs_fb(df_te, min_pos=max(1, min_each // 5), max_pairs=n_pairs)
            res_long, pair_summary = fb_bootstrap_summary(
                df_te, probs, fb_pairs, thr=thr, B=B_gap
            )
            pairs = fb_pairs  # keep consistent for n_pairs in info

    mean_gap = float(pair_summary["gap_mean"].mean()) if (pair_summary is not None and len(pair_summary)) else np.nan
    p90_gap  = float(pair_summary["gap_mean"].quantile(0.9)) if (pair_summary is not None and len(pair_summary)) else np.nan

    info = {
        "attr": attr_name,
        "layer": layer,
        "n_train": int(len(df_tr)),
        "n_test": int(len(df_te)),
        "train_pos_rate": float(ytr.mean()) if len(ytr) else np.nan,
        "test_pos_rate": float(yte.mean()) if len(yte) else np.nan,
        "test_acc": float(test_acc),
        "thr": float(thr),
        "epochs": int(epochs),
        "n_pairs": int(len(pairs)),
        "B_gap": int(B_gap),
        "B_species": int(B_species),
        "mean_gap": mean_gap,
        "p90_gap": p90_gap,
        "use_balanced_acc": bool(use_balanced_acc),
    }

    return info, res_long, pair_summary, df_te, species_table


## 4. Attribute screening

`screen_attributes_for_species_variation` tests all 26 FunnyBirds concepts and selects
those with meaningful cross-species recall variation.

**FunnyBirds-tuned thresholds (500 test images, 10 per class):**
- `min_pos_per_species=3` — each positive species has all 10 images positive; trivially ≥3
- `min_species_with_pos=4` — tail has ~5-6 species per variant; 4 is safe floor
- `min_overall_prev=0.01, max_overall_prev=0.99` — tail variants cover ~10% prevalence

Uses CBM features for screening (most informative); falls back to baseline if unavailable.


In [ ]:
def screen_attributes_for_species_variation(
    candidate_attrs,
    feat_dir: Path,
    kind_tr: str, ids_tr: np.ndarray,
    kind_te: str, ids_te: np.ndarray,
    layer: str,
    *,
    min_certainty: int = 1,
    thr: float = 0.5,
    min_pos_per_species: int = 10,
    min_species_with_pos: int = 15,
    min_overall_prev: float = 0.05,
    max_overall_prev: float = 0.95,
    epochs: int = 8,
    max_attrs: int | None = None,
    verbose_every: int = 50,
    keep_error_examples: int = 5,
    B_species: int = 200,   # NEW: bootstrap trials for species recall CI during screening
):
    rows = []
    errors = []
    stats = {
        "tried": 0,
        "success": 0,
        "filtered_too_few_species_pos": 0,
        "filtered_prev_out_of_range": 0,
        "filtered_no_recall_vals": 0,
        "errored": 0,
    }

    cand = list(candidate_attrs)
    if max_attrs is not None:
        cand = cand[:max_attrs]

    for i, attr in enumerate(cand):
        stats["tried"] += 1
        try:
            info, _, _, _, species_table = run_one_attribute(
                attr,
                feat_dir,
                kind_tr, ids_tr,
                kind_te, ids_te,
                layer=layer,
                min_certainty=min_certainty,
                thr=thr,
                epochs=epochs,
                min_each=10,
                n_pairs=0,          # screening: skip matched pairs
                B_species=B_species,
            )

            st = species_table.copy()
            overall_prev = float(st["n_pos"].sum() / st["n"].sum()) if st["n"].sum() > 0 else np.nan

            st_pos = st[st["n_pos"] >= min_pos_per_species].copy()
            n_species_pos = int(len(st_pos))

            if n_species_pos < min_species_with_pos:
                stats["filtered_too_few_species_pos"] += 1
                continue

            if not (min_overall_prev <= overall_prev <= max_overall_prev):
                stats["filtered_prev_out_of_range"] += 1
                continue

            recall_vals = st_pos["recall"].dropna().to_numpy()
            if recall_vals.size == 0:
                stats["filtered_no_recall_vals"] += 1
                continue

            stats["success"] += 1

            recall_std = float(np.std(recall_vals))
            recall_range = float(np.max(recall_vals) - np.min(recall_vals))
            recall_p90_p10 = float(np.quantile(recall_vals, 0.9) - np.quantile(recall_vals, 0.1))

            rows.append({
                "attr": attr,
                "overall_prev": overall_prev,
                "n_species_pos": n_species_pos,
                "recall_std": recall_std,
                "recall_range": recall_range,
                "recall_p90_p10": recall_p90_p10,
                "test_acc": float(info["test_acc"]),
                "n_test": int(info["n_test"]),
            })

            if verbose_every and ((i + 1) % verbose_every == 0):
                print(f"[{i+1}/{len(cand)}] ok: {attr}  prev={overall_prev:.3f}  n_species_pos={n_species_pos}")

        except Exception as e:
            stats["errored"] += 1
            if len(errors) < keep_error_examples:
                errors.append((attr, repr(e)))
            continue

    screen_df = pd.DataFrame(rows)

    print("\n--- Screening summary ---")
    for k, v in stats.items():
        print(f"{k}: {v}")
    if errors:
        print("\nExample errors (first few):")
        for a, msg in errors:
            print(" ", a, "->", msg)

    if screen_df.empty:
        print("\nNo attributes passed filters. Likely causes:")
        print(" - run_one_attribute is erroring for most attrs (see errors above)")
        print(" - filters too strict for your attribute distribution")
        return screen_df

    screen_df = screen_df.sort_values(
        ["recall_p90_p10", "recall_range", "recall_std"],
        ascending=False
    ).reset_index(drop=True)

    return screen_df


# ---- Run it ----
CANDIDATE_ATTRS = attr_df["attr_name"].tolist()

# FunnyBirds-tuned thresholds:
#   - 500 test images, 10 per class → each positive species has all 10 images positive
#   - tail has 9 variants → ~5-6 species per variant → min_species_with_pos must be ≤5
#   - concepts are perfectly balanced → no prevalence filter needed
_feat_dir = FB_FEATS[0.0] if FB_FEATS[0.0].exists() else CBM_FEATS

screen_df = screen_attributes_for_species_variation(
    CANDIDATE_ATTRS,
    feat_dir=_feat_dir,
    kind_tr=fb_kind_tr, ids_tr=fb_ids_tr,
    kind_te=fb_kind_te, ids_te=fb_ids_te,
    layer=LAYER,
    min_certainty=1,
    thr=0.5,
    min_pos_per_species=3,       # 10 test images/class → trivially ≥3
    min_species_with_pos=4,      # tail: ~5-6 species/variant; 4 is safe floor
    min_overall_prev=0.01,       # tail variants cover ~10% prevalence
    max_overall_prev=0.99,
    epochs=8,
    max_attrs=200,
    verbose_every=5,
)

print(f"\nScreened: {len(screen_df)} / {len(CANDIDATE_ATTRS)} concepts passed")

if not screen_df.empty:
    TOP_K = 26          # FunnyBirds: use all concepts that pass
    ATTR_LIST = screen_df["attr"].head(TOP_K).tolist()
    print("\nATTR_LIST:")
    for a in ATTR_LIST:
        print(" ", a)
    display(screen_df.head(30))

# Map: attr -> typical cross-species recall spread (from screening)
attr_to_spread = screen_df.set_index("attr")["recall_p90_p10"].to_dict() if not screen_df.empty else {}


In [ ]:
# run_many: loops over attr_list, calls run_one_attribute for each,
# collects (info_df, pairs_df, species_df).
# Verbatim from recall.ipynb — do not modify.

def run_many(
    attr_list,
    model_name,
    feat_dir,
    kind_tr, ids_tr,
    kind_te, ids_te,
    *,
    layer,
    min_certainty=1,
    thr=0.5,
    epochs=25,
    min_each=10,
    n_pairs=200,
    B_gap=100,
    B_species=100,
    use_balanced_acc=False,
):
    """
    Runs run_one_attribute over a list of attrs.
    Collects:
      - info_df: one row per attr (headline metrics)
      - pairs_df: per-(attr, pair) summary table (gap_mean, CI, p, etc.)
      - species_df: per-(attr, species) table (prevalence, recall, recall CI)
    """
    all_info = []
    all_pair_summ = []
    all_species = []

    for attr in attr_list:
        info, _, pair_summ, _, species_table = run_one_attribute(
            attr, feat_dir,
            kind_tr, ids_tr,
            kind_te, ids_te,
            layer=layer,
            min_certainty=min_certainty,
            thr=thr,
            epochs=epochs,
            min_each=min_each,
            n_pairs=n_pairs,
            B_gap=B_gap,
            B_species=B_species,
            use_balanced_acc=use_balanced_acc,
        )

        info = dict(info)
        info["model"] = model_name
        all_info.append(info)

        if pair_summ is not None and len(pair_summ):
            ps = pair_summ.copy()
            ps["attr"] = attr
            ps["model"] = model_name
            all_pair_summ.append(ps)

        st = species_table.copy()
        st["attr"] = attr
        st["model"] = model_name
        all_species.append(st)

        acc_label = "bal_acc" if use_balanced_acc else "test_acc"
        print(model_name, attr, f"{acc_label}=", round(info["test_acc"], 4), "mean_gap=", round(info["mean_gap"], 4))

    info_df = pd.DataFrame(all_info)
    pairs_df = pd.concat(all_pair_summ, ignore_index=True) if all_pair_summ else pd.DataFrame()
    species_df = pd.concat(all_species, ignore_index=True) if all_species else pd.DataFrame()
    return info_df, pairs_df, species_df


## 5. Matched-pair recall gap — Baseline vs CBM

For each concept, we compare:
- **Probe accuracy**: is the concept linearly decodable at `layer4.0`?
- **Cross-species recall gap**: does the probe recall the concept differently across species?

FunnyBirds pairing: concepts are 100% present or absent per species, so we pair
positive-concept species (C=1) vs negative-concept species (C=0) and measure
`gap = recall_positive − FPR_negative`. A large gap means the probe is well-separated.


In [ ]:
import numpy as np

# ── Screen concepts using whichever features are available ───────────────
_screen_feat = CBM_FEATS if CBM_FEATS.exists() else BASE_FEATS
_sk_tr = cbm_kind_tr if CBM_FEATS.exists() else base_kind_tr
_si_tr = cbm_ids_tr  if CBM_FEATS.exists() else base_ids_tr
_sk_te = cbm_kind_te if CBM_FEATS.exists() else base_kind_te
_si_te = cbm_ids_te  if CBM_FEATS.exists() else base_ids_te

screen_df = screen_attributes_for_species_variation(
    ATTR_LIST,
    feat_dir=_screen_feat,
    kind_tr=_sk_tr, ids_tr=_si_tr,
    kind_te=_sk_te, ids_te=_si_te,
    layer=LAYER,
    min_certainty=1, thr=0.5,
    min_pos_per_species=3,   # FunnyBirds: 10 test images/class
    min_species_with_pos=4,  # tail: ~5-6 species/variant
    min_overall_prev=0.01,
    max_overall_prev=0.99,
    epochs=8, max_attrs=200, verbose_every=5,
)
print(f'Screened: {len(screen_df)} / {len(ATTR_LIST)} concepts passed')
ATTR_LIST_SCREENED = screen_df['attr'].tolist() if len(screen_df) > 0 else ATTR_LIST
if len(screen_df) > 0:
    display(screen_df.head(30))


In [ ]:
results = {}

if BASE_FEATS.exists():
    print('Running baseline...')
    info_base, pairs_base, species_base = run_many(
        ATTR_LIST_SCREENED, 'baseline_fb', BASE_FEATS,
        base_kind_tr, base_ids_tr,
        base_kind_te, base_ids_te,
        layer=LAYER, thr=0.5, n_pairs=200, B_gap=100, B_species=100,
    )
    results['baseline'] = (info_base, pairs_base, species_base)
    species_base.to_csv('fb_baseline_species.csv', index=False)
    print(f'Baseline: {len(info_base)} concepts, mean gap={info_base.mean_gap.mean():.4f}')
else:
    print('[warn] Baseline features not found. Extract with --plain_resnet flag.')

if CBM_FEATS.exists():
    print('Running CBM...')
    info_cbm, pairs_cbm, species_cbm = run_many(
        ATTR_LIST_SCREENED, 'cbm_fb', CBM_FEATS,
        cbm_kind_tr, cbm_ids_tr,
        cbm_kind_te, cbm_ids_te,
        layer=LAYER, thr=0.5, n_pairs=200, B_gap=100, B_species=100,
    )
    results['cbm'] = (info_cbm, pairs_cbm, species_cbm)
    species_cbm.to_csv('fb_cbm_species.csv', index=False)
    print(f'CBM: {len(info_cbm)} concepts, mean gap={info_cbm.mean_gap.mean():.4f}')
else:
    print('[warn] CBM features not found. Run run_fb_cbm_adroit.sh first.')


In [ ]:
import pandas as pd

rows = []
for name, (info, _, _) in results.items():
    rows.append({'model': name, 'n_concepts': len(info),
                 'mean_gap': info['mean_gap'].mean(),
                 'frac_gap_gt0': float((info['mean_gap'] > 0).mean()),
                 'mean_probe_acc': info['test_acc'].mean()})
if rows:
    display(pd.DataFrame(rows))


In [ ]:
if 'baseline' in results and 'cbm' in results:
    info_b, _, _ = results['baseline']
    info_c, _, _ = results['cbm']
    t = (info_b[['attr','test_acc','mean_gap']]
         .rename(columns={'test_acc':'base_acc','mean_gap':'base_gap'})
         .merge(info_c[['attr','test_acc','mean_gap']]
                .rename(columns={'test_acc':'cbm_acc','mean_gap':'cbm_gap'}),
                on='attr', how='outer'))
    t['delta_gap'] = t['cbm_gap'] - t['base_gap']
    t['delta_acc'] = t['cbm_acc'] - t['base_acc']
    t = t.sort_values('base_gap', ascending=False)
    display(t)
    t.to_csv('fb_baseline_vs_cbm_table.csv', index=False)
    print('Saved fb_baseline_vs_cbm_table.csv')
elif len(results) == 1:
    name = list(results.keys())[0]
    info, _, _ = results[name]
    print(f'Only {name} available. Top concepts by recall gap:')
    display(info[['attr','test_acc','mean_gap','p90_gap','n_pairs']]
            .sort_values('mean_gap', ascending=False).head(26))


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if 'baseline' in results and 'cbm' in results:
    info_b, _, _ = results['baseline']
    info_c, _, _ = results['cbm']
    merged = (info_b[['attr','mean_gap','test_acc']]
              .rename(columns={'mean_gap':'gap_base','test_acc':'acc_base'})
              .merge(info_c[['attr','mean_gap','test_acc']]
                     .rename(columns={'mean_gap':'gap_cbm','test_acc':'acc_cbm'}),
                     on='attr', how='outer'))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    ax.scatter(merged['gap_base'], merged['gap_cbm'], alpha=0.7, s=50)
    for _, row in merged.dropna().iterrows():
        ax.annotate(row['attr'], (row['gap_base'], row['gap_cbm']), fontsize=6, alpha=0.6)
    lim = max(merged[['gap_base','gap_cbm']].max().max(), 0.05) * 1.1
    ax.plot([0,lim],[0,lim], 'k--', alpha=0.4, label='y=x')
    ax.set_xlabel('Baseline recall gap'); ax.set_ylabel('CBM recall gap')
    ax.set_title('Recall gap: Baseline vs CBM\n(above diagonal = CBM more entangled)')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.scatter(merged['acc_base'], merged['acc_cbm'], alpha=0.7, s=50)
    for _, row in merged.dropna().iterrows():
        ax.annotate(row['attr'], (row['acc_base'], row['acc_cbm']), fontsize=6, alpha=0.6)
    ax.plot([0,1],[0,1], 'k--', alpha=0.4, label='y=x')
    ax.set_xlabel('Baseline probe accuracy'); ax.set_ylabel('CBM probe accuracy')
    ax.set_title('Probe accuracy: Baseline vs CBM\n(above diagonal = CBM more concept-linear)')
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.suptitle('FunnyBirds: Baseline vs CBM', y=1.02)
    plt.tight_layout()
    plt.savefig('fb_baseline_vs_cbm.png', dpi=150, bbox_inches='tight')
    plt.show(); print('Saved fb_baseline_vs_cbm.png')


## Notes

### Hypothesis
CBM explicitly supervises the 26-concept bottleneck. Expected direction:
- **Baseline recall gap > CBM recall gap**: CBM disentangles species identity from concepts
- Probe accuracy: CBM ≥ baseline (concept supervision makes concepts more linearly decodable)

### FunnyBirds pairing strategy
Standard CUB-style balanced pairs (same pos:neg ratio per species) don't work here
because FunnyBirds concepts are 100% present or 100% absent per species.
We instead pair positive-concept species vs negative-concept species and measure:
- `recall_positive` = probe recall on positive-concept species images
- `FPR_negative` = false positive rate on negative-concept species images
- `gap = recall_positive − FPR_negative` (large = well-separated probe)

### Next step
Compare baseline > cbm > mcbm in `funnybirds_recall.ipynb` once MCBM features are ready.
